In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 

import sys
sys.path.append('/home/cat/code/bmi/')



#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, get_footprints_from_suite2p)


#from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
#                         correct_drift_single_frame, template_generation, 
 #                        plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0

from utils.calcium import calcium


Operating system:  Linux


Autosaving every 180 seconds


In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
fname = r'F:\bmi\andres\Cohort 7\DON-014451\day0\calibration\Image_001_001.raw'
fname = '/media/cat/4TB/donato/andres/Cohort7/14250/day0/plane0/Image_001_001.raw'


# 
bmi_c = CalibrationTools(fname)
bmi_c.calcium = calcium
#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations

#
bmi_c.std_map = bmi_c.data[:1000].mean(0)

# load suite2p footprints
bmi_c = get_footprints_from_suite2p(bmi_c)

# 
print ("DONE...")


memmap :  (90000, 512, 512)
  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (298, 90000)
         self.Fneu (neuropile):  (298, 90000)
         self.iscell (Suite2p cell classifier output):  (347, 2)
              of which number of good cells:  (298,)
         self.spks (deconnvoved spikes):  (298, 90000)
         self.stat (footprints structure):  (298,)
         mean std over all cells :  44.0
# of footprints;  298
DONE...


In [7]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
order_type = 'snr'  # 'snr' or 'f0'
bmi_c.compute_roi_traces_f0_and_reorder_cells(order_type)  # this function also coputes the snr / f0s of the cells

# 


computing roi traces for SNR indexing: 100%|█████████████████████████████████████| 9000/9000 [00:17<00:00, 517.00it/s]


In [9]:
print (bmi_c.roi_f0s)

plt.figure()
plt.hist(bmi_c.roi_f0s)
plt.show()

[833.9429, 509.9245, 884.10406, 746.04083, 653.7694, 554.2449, 565.4102, 1093.4653, 782.4959, 776.651, 739.3143, 899.62244, 661.53265, 580.3714, 776.0714, 793.75714, 783.37756, 650.1326, 540.7061, 831.10815, 627.4224, 756.4306, 871.5571, 610.0449, 846.68774, 552.17346, 899.16125, 766.33875, 913.8143, 741.45917, 706.8408, 613.551, 751.2204, 626.95715, 889.98773, 633.66736, 646.1653, 844.2225, 697.53064, 833.46124, 747.0612, 738.3898, 753.09796, 726.17957, 858.2184, 744.6959, 520.9327, 692.8041, 535.7122, 680.7694, 680.5633, 817.5204, 657.45917, 648.5041, 768.0102, 551.18164, 682.2225, 741.1245, 753.0694, 817.149, 660.002, 541.32855, 722.9224, 762.5102, 676.5082, 622.2082, 557.1326, 682.5204, 834.9918, 698.0796, 869.4224, 796.45306, 652.8449, 748.19183, 769.61835, 606.76324, 620.95105, 481.56326, 463.43674, 739.03674, 871.9388, 647.2857, 745.2939, 654.4959, 689.2714, 701.62244, 625.6449, 483.43265, 530.46326, 642.6551, 858.2959, 625.4224, 710.0082, 700.3755, 797.2225, 841.26733, 415.0591

In [60]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
bmi_c.scale=1.5
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 10       # Subsample the time series to go faster;

#
print (bmi_c.data.shape)
#mean = bmi_c.data[:1000].mean(0) #/ bmi_c.data[:1000].std(0)
#bmi_c.std_map = bmi_c.data[:1000].mean(0) #/ bmi_c.data[:1000].std(0)
bmi_c.std_map = np.median(bmi_c.data[::30], axis=0) #/ bmi_c.data[:1000].std(0)
#bmi_c.std_map = mean/bmi_c.std_map
#bmi_c.std_map = bmi_c.data[:1000].std(0)/bmi_c.std_map
bmi_c.std_map = bmi_c.std_map - np.mean(bmi_c.std_map)
# res = scipy.stats.mode(np.float32(bmi_c.data[:1000]), axis=0) #.std(0)/
# bmi_c.std_map = res[0].squeeze()
print (bmi_c.std_map.shape)
# visualize traces
bmi_c.vmin=0
bmi_c.vmax= 50
bmi_c.max_n_cells = 1000
bmi_c.min_f0 = 0
bmi_c.visualize_traces_snr_order(bmi_c.std_map)

(90000, 512, 512)
(512, 512)
memmap :  (90000, 512, 512)


In [42]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [27,16]
bmi_c.ensemble2 = [1,6]
bmi_c.both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", bmi_c.both)

#
# bmi_c.show_traces_ids(bmi_c.both)
# # ######################################################################
# # ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# # ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles2(bmi_c.std_map)

print ("DONE...")


all cells: [27 16  1  6]


100%|███████████████████████████████████████████████████████████████████████████| 90000/90000 [01:17<00:00, 1158.16it/s]


Contour:  [143 312]
Contour:  [133 403]
Contour:  [276 194]
cell ids:  [27 16  1  6]
DONE...


In [43]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 3   # reward lockout in seconds
bmi_c.post_missed_reward_lockout = 10
bmi_c.post_reward_lockout_baseline_min = 3
bmi_c.trial_time = 30
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.33#*0.85

#gr.find_reward_thresholds_low_and_high()
#bmi_c.high = 2
bmi_c.find_reward_thresholds_high_realtime()  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


# do not change this
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


nsec recording:  3000 max # of random rewards (i.e. every 30sec)  100
 @0.33% reward rate:  33
 high guess:  4.001189946630476
updated rewards #:  0 4.001189946630476
updated rewards #:  0 3.9611780471641715
updated rewards #:  0 3.9215662666925297
updated rewards #:  0 3.8823506040256044
updated rewards #:  0 3.843527097985348
updated rewards #:  0 3.8050918270054948
updated rewards #:  1 3.76704090873544
updated rewards #:  1 3.7293704996480854
updated rewards #:  1 3.6920767946516047
updated rewards #:  1 3.6551560267050887
updated rewards #:  3 3.618604466438038
updated rewards #:  3 3.5824184217736574
updated rewards #:  3 3.546594237555921
updated rewards #:  3 3.5111282951803617
updated rewards #:  3 3.4760170122285583
updated rewards #:  3 3.441256842106273
updated rewards #:  3 3.40684427368521
updated rewards #:  4 3.372775830948358
updated rewards #:  4 3.3390480726388745
updated rewards #:  4 3.3056575919124858
updated rewards #:  5 3.2726010159933607
updated rewards #:  5 

In [44]:
#############################################
########### SAVE DATA #######################
#############################################

#    
text = "day0"
bmi_c = save_calibration_data(bmi_c,text)  

contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
 couldn't save bmi_c.object .... TO FIX!
Done...
